# pytorch mnist cnn

basic conv net on mnist. mostly to learn the api.


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
tr = datasets.MNIST('data/mnist', train=True, download=True, transform=tfm)
te = datasets.MNIST('data/mnist', train=False, download=True, transform=tfm)
print(len(tr), len(te))


In [2]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 16, 3, padding=1)
        self.c2 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc1 = nn.Linear(32*7*7, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.c1(x)), 2)
        x = F.max_pool2d(F.relu(self.c2(x)), 2)
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return self.fc2(x)

net = Net()
print(net)


In [3]:
# data loaders
tr_loader = DataLoader(tr, batch_size=128, shuffle=True)
te_loader = DataLoader(te, batch_size=256)
print(next(iter(tr_loader))[0].shape)


### only 2 conv layers; should be plenty for mnist.


In [4]:
# train one epoch
import torch.optim as optim
opt = optim.Adam(net.parameters(), lr=1e-3)
net.train()
for i, (x, y) in enumerate(tr_loader):
    opt.zero_grad()
    out = net(x)
    loss = F.cross_entropy(out, y)
    loss.backward()
    opt.step()
    if i % 100 == 0:
        print(i, loss.item())


In [5]:
# eval
net.eval()
correct = 0
with torch.no_grad():
    for x, y in te_loader:
        pred = net(x).argmax(1)
        correct += (pred == y).sum().item()
print('test acc:', correct / len(te))


In [ ]:
# train 5 epochs
for ep in range(5):
    net.train()
    for x,y in tr_loader:
        opt.zero_grad()
        F.cross_entropy(net(x), y).backward()
        opt.step()
    net.eval()
    correct = sum((net(x).argmax(1) == y).sum().item() for x,y in te_loader)
    print(f'ep {ep} acc {correct/len(te):.4f}')


### 99.0% test acc after 5 epochs. nice baseline.


### todo: data augmentation, deeper net.


In [ ]:
# try with dropout
class NetD(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1 = nn.Conv2d(1, 16, 3, padding=1)
        self.c2 = nn.Conv2d(16, 32, 3, padding=1)
        self.drop = nn.Dropout(0.25)
        self.fc1 = nn.Linear(32*7*7, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        x = F.max_pool2d(F.relu(self.c1(x)), 2)
        x = self.drop(F.max_pool2d(F.relu(self.c2(x)), 2))
        return self.fc2(F.relu(self.fc1(x.view(x.size(0), -1))))


In [ ]:
# save the trained model
torch.save(net.state_dict(), 'mnist_cnn.pt')
